# Grid World Simulation for Quasi-2D Perovskite Synthesis Optimization

Converts the N1/N2/N3 campaign experimental data into a **discrete navigable grid** where:
- **X-axis** → Precursor ratio R_BAAc (butylammonium / lead)
- **Y-axis** → Annealing Temperature (°C)
- **Reward map** → Interpolated QW phase fractions from real measurements

A Q-learning agent learns to navigate this grid to find synthesis conditions that maximize a target quantum-well distribution.

---
**Data summary:**
- N1 Campaign: 78 points (R0 @ 60/105/150°C + R1 + R2)
- N2 Campaign: 216 points
- N3 Campaign: 162 points
- **Total: ~456 data points** used to build the reward map

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import RBFInterpolator

np.random.seed(42)
print('Libraries loaded.')

## 1. Load & Combine All Campaign Data

In [ ]:
# Notebook lives in .../experiments/ so paths are relative from there
BASE = '.'

# N1: .txt is actually comma-separated
with open(f'{BASE}/N1_Campaign/Data_Tracked/N1Campaign_data.txt') as f:
    df1 = pd.read_csv(io.StringIO(f.read()))
df1['campaign'] = 'N1'

df2 = pd.read_csv(f'{BASE}/N2_Campaign/Data_Tracked/N2Campaign_data.csv')
df2['campaign'] = 'N2'

df3 = pd.read_csv(f'{BASE}/N3_Campaign/Data_Tracked/N3Campaign_data.csv')
df3['campaign'] = 'N3'

df = pd.concat([df1, df2, df3], ignore_index=True)

QW_COLS    = [f'QW {i}' for i in range(1, 13)] + ['QW 99']
INPUT_COLS = ['Anneal Time', 'Temperature', 'R PbI2', 'R BAAc', 'R MAI']

df = df.dropna(subset=INPUT_COLS + QW_COLS).reset_index(drop=True)

print(f'Combined dataset: {len(df)} rows')
print(f'Campaign breakdown: {df.campaign.value_counts().to_dict()}')
print()
print(df[INPUT_COLS].describe().round(3))

## 2. Define the Reward Function

The reward at each synthesis condition is a **weighted sum of QW fractions**.

Change `QW_WEIGHTS` to target different phase distributions:
- **Low-n dominant (LEDs):** weight QW 1–3 heavily
- **Mid-n (solar):** weight QW 3–6
- **Minimize defects:** set `QW 99` to `-1.0`
- **Pure phase target:** set one QW to `1.0`, rest to `0`

In [ ]:
# ---------------------------------------------------------------
# CONFIGURE: set target QW distribution here
# ---------------------------------------------------------------
QW_WEIGHTS = {
    'QW 3':  1.0,
    'QW 4':  1.0,
    'QW 5':  0.5,
    'QW 99': -2.0,   # penalize defect fraction
}
REWARD_LABEL = 'QW3+QW4 (penalise defects)'

weights   = np.array([QW_WEIGHTS.get(c, 0.0) for c in QW_COLS])
df['reward'] = df[QW_COLS].values @ weights

print(f'Reward range: [{df.reward.min():.3f}, {df.reward.max():.3f}]  mean={df.reward.mean():.3f}')
print(f'Reward label: {REWARD_LABEL}')

## 3. Build the 20×20 Grid & Interpolate the Reward Map

In [ ]:
# ---------------------------------------------------------------
# CONFIGURE: grid axes
# ---------------------------------------------------------------
GRID_SIZE = 20
X_PARAM   = 'R BAAc'       # X-axis
Y_PARAM   = 'Temperature'  # Y-axis
X_MIN, X_MAX = 0.4,  2.5
Y_MIN, Y_MAX = 55.0, 155.0

x_ticks = np.linspace(X_MIN, X_MAX, GRID_SIZE)
y_ticks = np.linspace(Y_MIN, Y_MAX, GRID_SIZE)
xx, yy  = np.meshgrid(x_ticks, y_ticks)  # row=Y (temp), col=X (R_BAAc)

pts  = df[[X_PARAM, Y_PARAM]].values
vals = df['reward'].values

# Normalise to [0,1] for RBF numerical stability
def norm(v, lo, hi): return (v - lo) / (hi - lo)

pts_n  = np.column_stack([norm(pts[:,0], X_MIN, X_MAX),
                          norm(pts[:,1], Y_MIN, Y_MAX)])
grid_n = np.column_stack([norm(xx.ravel(), X_MIN, X_MAX),
                          norm(yy.ravel(), Y_MIN, Y_MAX)])

rbf        = RBFInterpolator(pts_n, vals, kernel='thin_plate_spline', smoothing=1.0)
reward_map = rbf(grid_n).reshape(GRID_SIZE, GRID_SIZE)

best_row, best_col = np.unravel_index(reward_map.argmax(), reward_map.shape)
print(f'Reward map: shape={reward_map.shape}  range=[{reward_map.min():.3f}, {reward_map.max():.3f}]')
print(f'Best cell: row={best_row} col={best_col} '
      f'-> T={y_ticks[best_row]:.1f}C, R_BAAc={x_ticks[best_col]:.3f}, '
      f'reward={reward_map[best_row,best_col]:.3f}')

## 4. Visualise the Reward Heatmap

In [ ]:
tick_idx = [0, 4, 9, 14, 19]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- continuous heatmap with raw data overlay ---
ax = axes[0]
im = ax.imshow(reward_map, origin='lower', aspect='auto',
               extent=[X_MIN, X_MAX, Y_MIN, Y_MAX], cmap='viridis')
plt.colorbar(im, ax=ax, label=f'Reward ({REWARD_LABEL})')

camp_colors = {'N1': 'red', 'N2': 'orange', 'N3': 'cyan'}
for camp, grp in df.groupby('campaign'):
    ax.scatter(grp[X_PARAM], grp[Y_PARAM], c=camp_colors[camp],
               s=20, edgecolors='black', linewidths=0.4,
               label=f'{camp} ({len(grp)})', zorder=5)
ax.scatter(x_ticks[best_col], y_ticks[best_row],
           marker='*', s=300, c='white', edgecolors='black',
           linewidths=1.5, zorder=10, label='Best cell')
ax.set_xlabel(f'X: {X_PARAM}')
ax.set_ylabel(f'Y: {Y_PARAM} (C)')
ax.set_title('Interpolated Reward Map (data points overlaid)')
ax.legend(fontsize=8)

# --- discrete grid ---
ax2 = axes[1]
im2 = ax2.imshow(reward_map, origin='lower', cmap='viridis', aspect='auto')
plt.colorbar(im2, ax=ax2, label='Reward')
for i in range(GRID_SIZE + 1):
    ax2.axhline(i - 0.5, color='white', linewidth=0.3, alpha=0.4)
    ax2.axvline(i - 0.5, color='white', linewidth=0.3, alpha=0.4)
ax2.plot(best_col, best_row, '*', markersize=18, color='white',
         markeredgecolor='black', markeredgewidth=1, zorder=10)
ax2.set_xticks(tick_idx)
ax2.set_xticklabels([f'{x_ticks[i]:.2f}' for i in tick_idx], fontsize=7)
ax2.set_yticks(tick_idx)
ax2.set_yticklabels([f'{y_ticks[i]:.0f}C' for i in tick_idx], fontsize=7)
ax2.set_xlabel(f'Grid col -> {X_PARAM}')
ax2.set_ylabel(f'Grid row -> {Y_PARAM}')
ax2.set_title(f'{GRID_SIZE}x{GRID_SIZE} Discrete Grid World  (* = best cell)')

plt.tight_layout()
plt.savefig('reward_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved reward_heatmap.png')

## 5. GridWorld Environment

| Action | Code | Effect |
|--------|------|--------|
| UP     | 0    | +Temperature |
| DOWN   | 1    | -Temperature |
| RIGHT  | 2    | +R_BAAc |
| LEFT   | 3    | -R_BAAc |
| WAIT   | 4    | Stay |

Each move costs a small **step penalty** — encourages the agent to reach high-reward regions efficiently.

In [ ]:
class GridWorldEnv:
    """
    Discrete grid world over the perovskite synthesis parameter space.
    State  : (row, col)
    Actions: 0=UP(+T), 1=DOWN(-T), 2=RIGHT(+BAAc), 3=LEFT(-BAAc), 4=WAIT
    """
    DELTAS       = {0: (-1,0), 1: (1,0), 2: (0,1), 3: (0,-1), 4: (0,0)}
    ACTION_NAMES = {0:'UP', 1:'DOWN', 2:'RIGHT', 3:'LEFT', 4:'WAIT'}

    def __init__(self, reward_map, x_ticks, y_ticks,
                 step_penalty=0.01, target_reward=None, max_steps=150):
        self.reward_map    = reward_map
        self.x_ticks       = x_ticks
        self.y_ticks       = y_ticks
        self.G             = reward_map.shape[0]
        self.step_penalty  = step_penalty
        self.target_reward = target_reward or 0.9 * reward_map.max()
        self.max_steps     = max_steps
        self.state         = None
        self.steps         = 0

    def reset(self, start=None):
        self.state = start if start is not None else (
            int(np.random.randint(self.G)), int(np.random.randint(self.G)))
        self.steps = 0
        return self.state

    def step(self, action):
        dr, dc     = self.DELTAS[action]
        r = int(np.clip(self.state[0] + dr, 0, self.G - 1))
        c = int(np.clip(self.state[1] + dc, 0, self.G - 1))
        self.state = (r, c)
        self.steps += 1
        cell_r  = float(self.reward_map[r, c])
        penalty = 0.0 if action == 4 else self.step_penalty
        done    = cell_r >= self.target_reward or self.steps >= self.max_steps
        info    = {'cell_reward': cell_r,
                   'real_x': float(self.x_ticks[c]),
                   'real_y': float(self.y_ticks[r])}
        return self.state, cell_r - penalty, done, info


env = GridWorldEnv(reward_map, x_ticks, y_ticks)
print(f'GridWorld ready.  Target reward threshold: {env.target_reward:.3f}')

## 6. Q-Learning Agent

Tabular Q-learning update rule:
$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]$$

In [ ]:
class QLearningAgent:
    def __init__(self, G, n_actions=5, lr=0.15, gamma=0.92,
                 eps=1.0, eps_min=0.05, eps_decay=0.995):
        self.Q         = np.zeros((G, G, n_actions))
        self.n_actions = n_actions
        self.lr        = lr
        self.gamma     = gamma
        self.eps       = eps
        self.eps_min   = eps_min
        self.eps_decay = eps_decay

    def act(self, state, greedy=False):
        if not greedy and np.random.random() < self.eps:
            return int(np.random.randint(self.n_actions))
        return int(np.argmax(self.Q[state[0], state[1]]))

    def update(self, s, a, r, ns, done):
        td = r + (0.0 if done else self.gamma * np.max(self.Q[ns[0], ns[1]])) \
             - self.Q[s[0], s[1], a]
        self.Q[s[0], s[1], a] += self.lr * td

    def decay(self):
        self.eps = max(self.eps_min, self.eps * self.eps_decay)


agent = QLearningAgent(GRID_SIZE)
print(f'Q-table shape: {agent.Q.shape}  ({GRID_SIZE}x{GRID_SIZE} states x 5 actions)')

## 7. Training Loop

In [ ]:
N_EPISODES = 3000
LOG_EVERY  = 300

ep_rewards, ep_steps, ep_best = [], [], []

for ep in range(N_EPISODES):
    state  = env.reset()
    total  = 0.0
    best_r = float(reward_map[state])

    while True:
        a               = agent.act(state)
        ns, r, done, info = env.step(a)
        agent.update(state, a, r, ns, done)
        total  += r
        best_r  = max(best_r, info['cell_reward'])
        state   = ns
        if done:
            break

    agent.decay()
    ep_rewards.append(total)
    ep_steps.append(env.steps)
    ep_best.append(best_r)

    if (ep + 1) % LOG_EVERY == 0:
        print(f'Ep {ep+1:4d} | avg_reward={np.mean(ep_rewards[-LOG_EVERY:]):.3f} '
              f'| avg_steps={np.mean(ep_steps[-LOG_EVERY:]):.1f} '
              f'| eps={agent.eps:.3f}')

print('Training complete.')

## 8. Training Curves

In [ ]:
W = 50
smooth = lambda x: np.convolve(x, np.ones(W)/W, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

axes[0].plot(smooth(ep_rewards), color='steelblue')
axes[0].set_title('Cumulative Reward (smoothed)'); axes[0].set_xlabel('Episode')

axes[1].plot(smooth(ep_steps), color='darkorange')
axes[1].set_title('Steps per Episode (smoothed)'); axes[1].set_xlabel('Episode')

axes[2].plot(smooth(ep_best), color='green', label='Agent best')
axes[2].axhline(reward_map.max(), color='red', linestyle='--', label='Global max')
axes[2].set_title('Best Cell Reward Reached per Episode')
axes[2].set_xlabel('Episode'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

## 9. Learned Policy Map

In [ ]:
ARROW  = {0:'↑', 1:'↓', 2:'→', 3:'←', 4:'·'}
policy = np.argmax(agent.Q, axis=2)  # shape (G, G)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
tick_idx  = [0, 4, 9, 14, 19]

ax = axes[0]
im = ax.imshow(reward_map, origin='lower', cmap='viridis', aspect='auto')
plt.colorbar(im, ax=ax, label='Reward')
for r in range(GRID_SIZE):
    for c in range(GRID_SIZE):
        ax.text(c, r, ARROW[policy[r,c]], ha='center', va='center',
                fontsize=7, color='white', fontweight='bold')
ax.plot(best_col, best_row, '*', markersize=16, color='yellow',
        markeredgecolor='black', markeredgewidth=1)
ax.set_xticks(tick_idx)
ax.set_xticklabels([f'{x_ticks[i]:.2f}' for i in tick_idx], fontsize=7)
ax.set_yticks(tick_idx)
ax.set_yticklabels([f'{y_ticks[i]:.0f}C' for i in tick_idx], fontsize=7)
ax.set_xlabel(X_PARAM); ax.set_ylabel(Y_PARAM)
ax.set_title('Learned Policy (greedy action per cell)')

ax2 = axes[1]
q_max = agent.Q.max(axis=2)
im2 = ax2.imshow(q_max, origin='lower', cmap='plasma', aspect='auto')
plt.colorbar(im2, ax=ax2, label='Max Q-value')
ax2.set_xticks(tick_idx)
ax2.set_xticklabels([f'{x_ticks[i]:.2f}' for i in tick_idx], fontsize=7)
ax2.set_yticks(tick_idx)
ax2.set_yticklabels([f'{y_ticks[i]:.0f}C' for i in tick_idx], fontsize=7)
ax2.set_xlabel(X_PARAM); ax2.set_ylabel(Y_PARAM)
ax2.set_title('Q-value Map (brighter = higher agent confidence)')

plt.tight_layout()
plt.savefig('policy_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved policy_map.png')

## 10. Greedy Trajectory Visualisation

In [ ]:
def run_greedy(env, agent, start=None):
    state = env.reset(start=start)
    traj, infos = [state], []
    while True:
        a = agent.act(state, greedy=True)
        ns, r, done, info = env.step(a)
        traj.append(ns)
        infos.append({**info, 'action': env.ACTION_NAMES[a]})
        state = ns
        if done: break
    return traj, infos

traj, infos = run_greedy(env, agent, start=(0, 0))
rows = [s[0] for s in traj]
cols = [s[1] for s in traj]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(reward_map, origin='lower', cmap='viridis', aspect='auto')
plt.colorbar(im, ax=ax, label='Reward')
ax.plot(cols, rows, 'w-o', markersize=5, linewidth=1.5, label='Agent path', zorder=6)
ax.plot(cols[0],  rows[0],  'gs', markersize=12, label='Start',      zorder=7)
ax.plot(cols[-1], rows[-1], 'r*', markersize=18, label='End',        zorder=7)
ax.plot(best_col, best_row, 'y*', markersize=18, label='Global max', zorder=8)
for i, (r, c) in enumerate(zip(rows, cols)):
    if i % max(1, len(rows)//8) == 0:
        ax.text(c + 0.3, r + 0.3, str(i), fontsize=7, color='white')
ax.set_xticks(tick_idx)
ax.set_xticklabels([f'{x_ticks[i]:.2f}' for i in tick_idx], fontsize=8)
ax.set_yticks(tick_idx)
ax.set_yticklabels([f'{y_ticks[i]:.0f}C' for i in tick_idx], fontsize=8)
ax.set_xlabel(X_PARAM); ax.set_ylabel(Y_PARAM)
ax.set_title(f'Greedy Episode from (0,0) — {len(traj)-1} steps')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

final = infos[-1]
print(f'Reached: T={final["real_y"]:.1f}C, R_BAAc={final["real_x"]:.3f}, reward={final["cell_reward"]:.4f}')
print(f'Global max: T={y_ticks[best_row]:.1f}C, R_BAAc={x_ticks[best_col]:.3f}, reward={reward_map[best_row,best_col]:.4f}')

## 11. QW Distribution at the Optimal Synthesis Condition

In [ ]:
optimal_x = x_ticks[best_col]
optimal_y = y_ticks[best_row]

# Euclidean distance in normalised space
dist = np.sqrt(
    ((df[X_PARAM] - optimal_x) / (X_MAX - X_MIN))**2 +
    ((df[Y_PARAM] - optimal_y) / (Y_MAX - Y_MIN))**2
)
nearest_idx = dist.nsmallest(5).index
nearest     = df.loc[nearest_idx]

print(f'Optimal grid cell: R_BAAc={optimal_x:.3f}, T={optimal_y:.1f}C')
print('\n5 nearest measured points:')
show_cols = ['campaign','Anneal Time','Temperature','R BAAc','R MAI','reward'] + QW_COLS[:6]
print(nearest[show_cols].to_string(index=False))

# Bar chart of QW fractions for the single closest point
best_pt  = nearest.iloc[0]
qw_vals  = best_pt[QW_COLS].values.astype(float)

fig, ax = plt.subplots(figsize=(9, 4))
bar_colors = plt.cm.coolwarm(np.linspace(0, 1, len(QW_COLS)))
ax.bar(QW_COLS, qw_vals, color=bar_colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Quantum Well Phase')
ax.set_ylabel('Fraction')
ax.set_title(
    f'QW Distribution @ Nearest Optimal Point\n'
    f'{best_pt.campaign} | T={best_pt["Temperature"]:.0f}C | '
    f'R_BAAc={best_pt["R BAAc"]:.3f} | Anneal={best_pt["Anneal Time"]:.0f}s'
)
ax.set_xticklabels(QW_COLS, rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('optimal_qw_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved optimal_qw_distribution.png')

## Summary

| Parameter | Value |
|-----------|-------|
| Total data points | N1 (78) + N2 (216) + N3 (162) = 456 |
| Grid size | 20×20 |
| X-axis | R_BAAc (0.4 → 2.5) |
| Y-axis | Temperature (55 → 155°C) |
| Interpolation | RBF thin-plate spline (smoothing=1) |
| RL algorithm | Tabular Q-learning |
| Training episodes | 3000 |
| Outputs | reward_heatmap.png, training_curves.png, policy_map.png, trajectory.png |

### Next Steps
- **3D grid:** add `Anneal Time` as a 3rd axis (requires DQN instead of tabular Q-table)
- **Change reward:** edit `QW_WEIGHTS` in Cell 3 to chase a different phase
- **DQN:** replace Q-table with a small MLP for function approximation
- **BO seeding:** use `reward_map` as a GP prior mean for the next BO campaign round